# Chapter 2 · Language modeling — Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/ch02_language_modeling.ipynb)

Reproduces **Chapter 2** live: the BPE trainer and its merges, why the first loss is always ln(V), the shape of cross-entropy, the loss↔perplexity dictionary, and one distribution sampled at three temperatures.

📖 Read the chapter: [Language modeling](https://omar-florez.github.io/training-efficient-llms/pdf/ch02_language_modeling.pdf)

In [ ]:
import numpy as np, matplotlib.pyplot as plt

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})
print("ready")

## 1 · BPE from scratch (the twelve-word corpus)

The complete trainer from Chapter 2.1 — greedy merges of the most frequent adjacent pair. Watch *-er*, *the*, and *fast* crystallize.

In [ ]:
from collections import Counter

def train_bpe(text, n_merges):
    words = Counter(text.split())
    vocab = {w: list(w) + ["</w>"] for w in words}
    merges, tokens_hist = [], []
    def count_tokens():
        return sum(len(vocab[w])*n for w, n in words.items())
    tokens_hist.append(count_tokens())
    for _ in range(n_merges):
        pairs = Counter()
        for w, n in words.items():
            for a, b in zip(vocab[w], vocab[w][1:]): pairs[(a, b)] += n
        if not pairs: break
        (a, b), n = pairs.most_common(1)[0]
        if n < 2: break
        for w in vocab:
            s, out, i = vocab[w], [], 0
            while i < len(s):
                if i < len(s)-1 and (s[i], s[i+1]) == (a, b): out.append(a+b); i += 2
                else: out.append(s[i]); i += 1
            vocab[w] = out
        merges.append((a+b, n)); tokens_hist.append(count_tokens())
    return vocab, merges, tokens_hist

corpus = "the faster runner ran past the fastest swimmer the swimmer swam faster"
vocab, merges, _ = train_bpe(corpus, 10)
print("merges:", [f"{m} x{n}" for m, n in merges])
for w in ["the", "faster", "fastest", "swimmer", "ran"]:
    print(f"{w:>10} -> {vocab[w]}")

In [ ]:
paragraph = ("language models eat numbers so a tokenizer must chop text into units and the units it learns "
             "depend only on frequency the most frequent pairs merge first and frequent words become single tokens "
             "while rare words stay in pieces this is compression and compression is compute saved") * 3
_, merges2, hist2 = train_bpe(paragraph, 60)

plt.figure(figsize=(8, 4))
plt.plot(range(len(hist2)), hist2, color=NAVY, lw=2)
plt.xlabel("merges performed"); plt.ylabel("tokens needed for the corpus")
plt.title(f"Compression as the vocabulary grows: {hist2[0]} → {hist2[-1]} tokens ({hist2[0]/hist2[-1]:.2f}×)")
plt.show()

## 2 · Cross-entropy: the shape of the loss

Pay −log p of the truth. Being confidently wrong is the worst sin — and an untrained model over V tokens pays exactly ln V, which is why every GPT-2-vocab run opens at **10.8**.

In [ ]:
p = np.linspace(0.001, 1, 500)
plt.figure(figsize=(8, 4))
plt.plot(p, -np.log(p), color=NAVY, lw=2)
for pt, lbl in [(0.7, "confident & right: 0.36"), (0.2, "uniform over 5: 1.61"), (0.02, "confidently wrong: 3.91")]:
    plt.plot(pt, -np.log(pt), "o", color=AMBER)
    plt.annotate(lbl, (pt, -np.log(pt)), xytext=(pt+0.05, -np.log(pt)+0.25), fontsize=9)
plt.xlabel("probability assigned to the correct token"); plt.ylabel("loss  = −log p")
plt.title("Cross-entropy punishes confident wrongness without bound")
plt.show()

for V in [50257, 32000, 128256]:
    print(f"untrained model, vocab {V:>7,}: first loss = ln(V) = {np.log(V):.2f}")

## 3 · The loss ↔ perplexity dictionary, and the shape of a run

Perplexity = e^loss: the effective number of choices the model hesitates between. A training curve is a **cliff** (cheap statistics) then a **grind** (the long tail) toward the entropy floor of language itself.

In [ ]:
import math
print(f"{'loss (nats)':>12} {'perplexity':>12}   meaning")
rows = [(math.log(50257), "uniform over GPT-2 vocab"), (6.9, "unigram statistics"), (4.0, "local grammar"),
        (3.0, "competent small model"), (2.3, "strong pretrained"), (1.7, "frontier territory")]
for L, note in rows:
    print(f"{L:>12.2f} {math.exp(L):>12,.1f}   {note}")

toks = np.logspace(6, 11.5, 400)                        # tokens seen (log axis)
loss = 1.75 + (10.82-1.75) * (toks/1e6) ** -0.32        # stylized power-law decay
plt.figure(figsize=(8, 4))
plt.semilogx(toks, loss, color=NAVY, lw=2)
plt.axhline(1.6, color=AMBER, ls="--", lw=1.2); plt.annotate("entropy of language (floor)", (2e6, 1.66), color=AMBER, fontsize=9)
plt.annotate("the cliff:\nfrequencies, grammar", (1.5e6, 6.5), fontsize=9, color=GRAY)
plt.annotate("the grind: rare facts, reasoning —\neach 0.01 costs more than the last", (2e9, 2.7), fontsize=9, color=GRAY)
plt.xlabel("training tokens (log)"); plt.ylabel("held-out loss")
plt.title("Every pre-training curve you will ever stare at (schematic)")
plt.show()

## 4 · Temperature: one model, three personalities

Table 2.2 recomputed and then *sampled*: the same five logits, reshaped by T, then 500 draws each.

In [ ]:
z = np.array([5.0, 3.0, 2.0, 1.0, 0.5])
temps = [0.5, 1.0, 2.0]
rng = np.random.default_rng(1)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
for ax, T in zip(axes, temps):
    prob = np.exp(z/T) / np.exp(z/T).sum()
    draws = rng.choice(5, size=500, p=prob)
    freq = np.bincount(draws, minlength=5) / 500
    xpos = np.arange(5)
    ax.bar(xpos-0.2, prob, 0.4, color=NAVY, label="p(token)")
    ax.bar(xpos+0.2, freq, 0.4, color=AMBER, label="500 samples")
    ax.set_title(f"T = {T}"); ax.set_xticks(xpos, [f"tok{i+1}" for i in xpos])
axes[0].set_ylabel("probability"); axes[0].legend(fontsize=9)
plt.suptitle("Sharpened (near-greedy) → as trained → adventurous", y=1.02)
plt.tight_layout(); plt.show()

## Break things

1. Train BPE on a paragraph in **Spanish** and one in English of the same meaning. Compare tokens needed — you just measured the fertility tax of Chapter 11.
2. In §3, change the power-law exponent. How many *more* tokens to go from loss 2.0 → 1.9 than from 3.0 → 2.9?
3. In §4, add top-k truncation before sampling and watch the tail disappear at high T.
4. Compute your own name's segmentation after 10 vs 60 merges of §1's paragraph corpus.